# 🍄 Boscdex Classifier Training on Google Colab (Free T4 GPU)

This notebook fine-tunes an on-device image classifier for **Boscdex** using **MobileNetV2** and **Transfer Learning**.

### 🚀 Highlights:
1. **1-Click Google Drive Ingestion:** Automatically finds and unzips `boscdex_dataset.zip` from your Google Drive into `./dataset/`.
2. **Hybrid / Dual Sourcing:** Uses your field photos from `test/assets/`, with optional iNaturalist expansion for underrepresented species.
3. **Decoupled Architecture:** Exports both the **TFLite feature extractor** (`boscdex_classifier.tflite`) and the **binary classification head** (`classifier_head.bin`), guaranteed 100% compatible with the Boscdex Flutter app!

### ⚡ Quick Start:
1. Ensure runtime is set to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Click **Runtime > Run all** (or run each cell sequentially).
3. When finished, your browser will automatically download the 3 updated model files!


In [ ]:
# Step 1: Verify Hardware & GPU Allocation
import sys
import platform
import shutil
import tensorflow as tf

print(f"Python Version:      {sys.version.split()[0]}")
print(f"Platform:            {platform.platform()}")
print(f"TensorFlow Version:  {tf.__version__}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"✅ GPU Acceleration Active: {gpus[0].name}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print("⚠️ No GPU detected. For 10x faster training, select: Runtime > Change runtime type > T4 GPU.")


In [ ]:
# Step 2: Install Required Libraries
!pip install -q pyinaturalist requests pillow tqdm


In [ ]:
# Step 3: Species Catalog & Directory Structure
import os
from pathlib import Path

DATASET_DIR = Path("./dataset")
EXPORT_DIR = Path("./export")
DATASET_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 20 Catalan Fungi Taxa in Boscdex (folder_id, inaturalist_taxon_id, catalan_name)
SPECIES_CATALOG = [
    ("lactarius_deliciosus", 54595, "Rovelló / Pinetell"),
    ("boletus_edulis", 48701, "Cep"),
    ("cantharellus_cibarius", 47347, "Rossinyol"),
    ("amanita_caesarea", 204588, "Ou de reig"),
    ("craterellus_cornucopioides", 48607, "Trompeta de la mort"),
    ("macrolepiota_procera", 63401, "Apagallums"),
    ("tricholoma_terreum", 63438, "Fredolic"),
    ("calocybe_gambosa", 122135, "Moixernó"),
    ("pleurotus_ostreatus", 48494, "Gírgola"),
    ("hydnum_repandum", 48682, "Llengua de bou"),
    ("hygrophorus_latitabundus", 353142, "Llenega negra"),
    ("craterellus_lutescens", 67707, "Camagroc"),
    ("suillus_luteus", 53488, "Llenegot"),
    ("russula_cyanoxantha", 63584, "Llora verda"),
    ("amanita_muscaria", 48715, "Reig bord"),
    ("omphalotus_olearius", 217916, "Gírgola d'olivera"),
    ("entoloma_sinuatum", 921907, "Pèrfid"),
    ("amanita_phalloides", 52135, "Farinera borda"),
    ("amanita_verna", 58694, "Cicuta blanca"),
    ("cortinarius_orellanus", 202404, "Cortinari de muntanya"),
    ("lactarius_sanguifluus", 118207, "Rovelló d'obaga"),
    ("chroogomphus_rutilus", 48714, "Cama de perdiu"),
]

for s_id, _, _ in SPECIES_CATALOG:
    (DATASET_DIR / s_id).mkdir(parents=True, exist_ok=True)

print(f"✓ Configured {len(SPECIES_CATALOG)} species directories in {DATASET_DIR.resolve()}")


In [ ]:
# Step 4: Dataset Ingestion (Google Drive boscdex_dataset.zip or local upload)
import zipfile
import shutil

# 1. Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"ℹ️ Google Drive mount note: {e}")

# 2. Check candidate locations for boscdex_dataset.zip
candidate_paths = [
    Path("/content/drive/MyDrive/boscdex_dataset.zip"),
    Path("/content/drive/MyDrive/boscdex/boscdex_dataset.zip"),
    Path("./boscdex_dataset.zip"),
]

found_archive = next((p for p in candidate_paths if p.exists()), None)

if found_archive:
    print(f"📦 Found dataset archive: {found_archive} ({found_archive.stat().st_size / (1024*1024):.1f} MB)")
    print("Extracting images into ./dataset/...")
    with zipfile.ZipFile(found_archive, "r") as zf:
        zf.extractall(DATASET_DIR)
    print("✅ Extraction complete!")
else:
    print("⚠️ No boscdex_dataset.zip found in Google Drive root or current working directory.")
    print("   Upload boscdex_dataset.zip to Google Drive (MyDrive/) or click the folder icon on the left to upload it.")

# 3. Print census
print()
print("📊 Dataset Census by Species:")
total_images = 0
for s_id, _, name in SPECIES_CATALOG:
    folder = DATASET_DIR / s_id
    count = len(list(folder.glob("*.*"))) if folder.exists() else 0
    total_images += count
    print(f"  • {s_id.ljust(28)}: {count:3d} images ({name})")
print(f"Total images available: {total_images}")


In [ ]:
# Step 5: Optional iNaturalist Top-Up for Balanced Classes
import time
import io
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import requests
from tqdm.notebook import tqdm
from pyinaturalist import get_observations

# Target minimum images per species:
# Set to 0 to train ONLY on your uploaded photos.
# Set to 30-50 to enrich rare classes with research-grade Catalan/Spanish photos.
MIN_IMAGES_PER_SPECIES = 35

PLACE_CATALONIA = 7161
PLACE_SPAIN = 6774
PLACE_EUROPE = 97391

def download_and_save(url, dest_path):
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            w, h = img.size
            min_dim = min(w, h)
            left = (w - min_dim) // 2
            top = (h - min_dim) // 2
            img = img.crop((left, top, left + min_dim, top + min_dim))
            img = img.resize((224, 224), Image.Resampling.LANCZOS)
            img.save(dest_path, "JPEG", quality=90)
            return True
    except Exception:
        pass
    return False

def top_up_species(s_id, taxon_id, target_count):
    folder = DATASET_DIR / s_id
    folder.mkdir(parents=True, exist_ok=True)
    existing = len(list(folder.glob("*.*")))
    if existing >= target_count:
        return existing

    downloaded = existing
    for pid in [PLACE_CATALONIA, PLACE_SPAIN, PLACE_EUROPE]:
        if downloaded >= target_count:
            break
        for page in range(1, 6):
            if downloaded >= target_count:
                break
            try:
                obs = get_observations(
                    taxon_id=taxon_id,
                    place_id=pid,
                    quality_grade="research",
                    photos=True,
                    per_page=50,
                    page=page,
                )
                results = obs.get("results", [])
                if not results:
                    break
                urls = []
                for o in results:
                    for p in o.get("photos", []):
                        u = p.get("url", "").replace("square", "medium")
                        if u:
                            urls.append(u)

                for u in urls:
                    if downloaded >= target_count:
                        break
                    downloaded += 1
                    dest = folder / f"inat_{downloaded:04d}.jpg"
                    if not download_and_save(u, dest):
                        downloaded -= 1
            except Exception:
                break
    return downloaded

if MIN_IMAGES_PER_SPECIES > 0:
    print(f"Checking that every species has at least {MIN_IMAGES_PER_SPECIES} images...")
    for s_id, t_id, name in tqdm(SPECIES_CATALOG, desc="Species Top-Up"):
        top_up_species(s_id, t_id, target_count=MIN_IMAGES_PER_SPECIES)
        time.sleep(0.2)
    print("✅ Dataset balanced and ready for training!")
else:
    print("Skipping iNaturalist download (MIN_IMAGES_PER_SPECIES=0).")


In [ ]:
# Step 6: Train & Fine-Tune MobileNetV2 (Native Keras Transfer Learning)
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU Acceleration:    {'✅ Available (' + gpus[0].name + ')' if gpus else '⚠️ CPU Only'}")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print("Loading dataset...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

class_names = train_ds.class_names
print(f"Classes ({len(class_names)}): {class_names}")

# Data augmentation pipeline
AUTOTUNE = tf.data.AUTOTUNE
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.20),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.10),
])

train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# Base MobileNetV2 pre-trained on ImageNet
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

# Architecture with Named Layers for Decoupled Extraction
# Note: mobilenet_v2.preprocess_input scales [0, 255] float input to [-1, 1] internally!
inputs = tf.keras.Input(shape=(224, 224, 3), name="input_image")
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
gap = tf.keras.layers.GlobalAveragePooling2D(name="gap_features")(x)
dropout = tf.keras.layers.Dropout(0.2, name="dropout")(gap)
head = tf.keras.layers.Dense(len(class_names), name="classifier_head")(dropout)
outputs = tf.keras.layers.Softmax(name="softmax")(head)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="boscdex_classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print()
print("🚀 Phase 1: Training top classification head (12 epochs)...")
model.fit(train_ds, validation_data=val_ds, epochs=12)

print()
print("🚀 Phase 2: Fine-tuning top MobileNetV2 layers (12 epochs)...")
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(train_ds, validation_data=val_ds, epochs=12)

val_loss, val_acc = model.evaluate(val_ds)
print()
print(f"📊 Final Model Validation Accuracy: {val_acc * 100:.2f}%")


In [ ]:
# Step 7: Export Decoupled Model Assets for Boscdex Flutter App
import struct
import json

EXPORT_DIR = Path("./export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export labels.txt
labels_path = EXPORT_DIR / "labels.txt"
with open(labels_path, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(f"{name}\n")
print(f"✓ Exported {len(class_names)} classes to: {labels_path}")

# 2. Export GAP Feature Extractor (Input: [1, 224, 224, 3], Output: [1, 1280] Float32)
# This matches TfliteClassifierService in Flutter exactly!
gap_model = tf.keras.Model(inputs=model.input, outputs=model.get_layer("gap_features").output)
converter = tf.lite.TFLiteConverter.from_keras_model(gap_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_gap = converter.convert()

tflite_path = EXPORT_DIR / "boscdex_classifier.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_gap)
print(f"✓ Exported GAP feature extractor: {tflite_path} ({tflite_path.stat().st_size / (1024*1024):.2f} MB)")

# 3. Export Portable Binary Classification Head (classifier_head.bin)
# Format: 20 float32 biases followed by 20 x 1280 float32 weights (little-endian)
dense_layer = model.get_layer("classifier_head")
weights, biases = dense_layer.get_weights()  # weights shape: (1280, 20), biases shape: (20,)

head_bin_path = EXPORT_DIR / "classifier_head.bin"
with open(head_bin_path, "wb") as f:
    # Write biases (20 floats)
    for b in biases:
        f.write(struct.pack("<f", float(b)))
    # Write weights (20 classes x 1280 features)
    for c in range(len(class_names)):
        for feature_idx in range(1280):
            w = weights[feature_idx, c]
            f.write(struct.pack("<f", float(w)))

print(f"✓ Exported binary classification head: {head_bin_path} ({head_bin_path.stat().st_size / 1024:.1f} KB)")

# 4. Also export full end-to-end model for reference
full_converter = tf.lite.TFLiteConverter.from_keras_model(model)
full_tflite = full_converter.convert()
full_path = EXPORT_DIR / "boscdex_classifier_full.tflite"
with open(full_path, "wb") as f:
    f.write(full_tflite)
print(f"✓ Exported full reference model: {full_path} ({full_path.stat().st_size / (1024*1024):.2f} MB)")


In [ ]:
# Step 8: Download Model Files Directly to Your Computer
try:
    from google.colab import files
    print("Triggering browser download of Boscdex model assets...")
    files.download(str(tflite_path))
    files.download(str(head_bin_path))
    files.download(str(labels_path))
    print()
    print("🎉 Done! Copy these 3 files into your local project:")
    print("   1. assets/models/boscdex_classifier.tflite")
    print("   2. assets/models/classifier_head.bin")
    print("   3. assets/models/labels.txt")
    print()
    print("Then verify with:")
    print("   flutter test test/unit_tests/domain/model_real_images_test.dart")
except Exception as e:
    print(f"Files ready in ./export/:")
    print(f" - {tflite_path}")
    print(f" - {head_bin_path}")
    print(f" - {labels_path}")
